# Notebook 3 — Transition Matrix Calibration

**Goal:** Combine the physical adjacency prior (Notebook 1) with observed review transitions (Notebook 2) to produce a calibrated floor-to-floor transition matrix for the ABM.

**Method:** Bayesian-inspired blending:
- **Prior:** Physical adjacency (adjacent floors have higher transition probability)
- **Likelihood:** Observed transitions from review NLP
- **Posterior:** Weighted blend, with higher weight on observations where data is sufficient

This approach is transparent: if we had real WiFi/camera data, we'd replace the prior entirely. The prior exists only to regularize sparse observations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-roastery-abm')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-roastery-abm')
    PROC_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../data/raw')
    PROC_DIR = Path('../data/processed')

floor_labels = ['1F', '2F', '3F', '4F', '5F']

# Load prior (physical adjacency)
prior = pd.read_csv(DATA_DIR / 'floor_adjacency_prior.csv', index_col=0)
prior_matrix = prior[floor_labels].values
print('Prior (physical adjacency):')
display(prior[floor_labels].round(3))

# Load observed transitions
observed = pd.read_csv(PROC_DIR / 'transition_matrix_observed.csv', index_col=0)
observed_matrix = observed.values
print('\nObserved (from review NLP):')
display(observed.round(3))

## Section 1 — Bayesian Blending

We blend prior and observed using a confidence weight `alpha`:

```
calibrated = alpha * observed + (1 - alpha) * prior
```

`alpha` is set per row based on the number of observed transitions from that floor. More observations → higher confidence in the data.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BAYESIAN BLENDING
# ══════════════════════════════════════════════════════════════════════

# Load raw transition counts to compute per-row confidence
raw_counts = np.zeros((5, 5))
reviews = pd.read_csv(DATA_DIR / 'reviews_manual.csv')
import re
FLOOR_PATTERNS = {
    '1F': r'first\s+floor|1st\s+floor|ground\s+floor|level\s+one|level\s+1\b|main\s+level|bottom\s+floor',
    '2F': r'second\s+floor|2nd\s+floor|level\s+two|level\s+2\b|bakery|princi|cafe\s+floor',
    '3F': r'third\s+floor|3rd\s+floor|level\s+three|level\s+3\b|experiential|experimental|experience\s+bar',
    '4F': r'fourth\s+floor|4th\s+floor|level\s+four|level\s+4\b|cocktail\s+bar|arriviamo',
    '5F': r'fifth\s+floor|5th\s+floor|rooftop|roof\s+top|terrace|patio',
}

def extract_seq(text):
    mentions = []
    for floor, pattern in FLOOR_PATTERNS.items():
        for match in re.finditer(pattern, text, re.IGNORECASE):
            mentions.append((match.start(), floor))
    mentions.sort()
    seq = []
    for _, f in mentions:
        if not seq or seq[-1] != f:
            seq.append(f)
    return seq

for _, row in reviews.iterrows():
    seq = extract_seq(row['text'])
    for i in range(len(seq)-1):
        fi = floor_labels.index(seq[i])
        fj = floor_labels.index(seq[i+1])
        raw_counts[fi][fj] += 1

# Compute alpha per row: sigmoid of observation count
# alpha = n_obs / (n_obs + k), where k is a smoothing constant
K_SMOOTH = 10  # With 10 observations, alpha = 0.5 (equal weight)
row_totals = raw_counts.sum(axis=1)
alpha_per_row = row_totals / (row_totals + K_SMOOTH)

print('=== Per-Floor Confidence ===')
for i, f in enumerate(floor_labels):
    print(f'  {f}: {int(row_totals[i])} transitions observed, alpha = {alpha_per_row[i]:.3f}')

# Blend
calibrated = np.zeros((5, 5))
for i in range(5):
    alpha = alpha_per_row[i]
    calibrated[i] = alpha * observed_matrix[i] + (1 - alpha) * prior_matrix[i]

# Add exit probability
exit_probs = 1.0 - calibrated.sum(axis=1)

# Ensure no negative exit probs (normalize if needed)
for i in range(5):
    if exit_probs[i] < 0.05:  # Minimum 5% exit probability
        calibrated[i] = calibrated[i] * (0.95 / calibrated[i].sum())
        exit_probs[i] = 0.05

calibrated_df = pd.DataFrame(calibrated.round(4), index=floor_labels, columns=floor_labels)
calibrated_df['exit'] = exit_probs.round(4)

print('\n=== Calibrated Transition Matrix ===')
display(calibrated_df)

## Section 2 — Visual Comparison: Prior vs Observed vs Calibrated

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# THREE-WAY COMPARISON
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = [prior_matrix, observed_matrix, calibrated]
titles = ['Prior (Physical Adjacency)', 'Observed (Review NLP)', 'Calibrated (Blended)']
cmaps = ['Blues', 'Oranges', 'Greens']

for ax, matrix, title, cmap in zip(axes, matrices, titles, cmaps):
    im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=0.6)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(floor_labels); ax.set_yticklabels(floor_labels)
    ax.set_xlabel('To'); ax.set_ylabel('From')
    ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{matrix[i][j]:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if matrix[i][j] > 0.3 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Transition Matrix: Prior → Observed → Calibrated', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Difference analysis
print('=== Where Observations Shifted the Prior Most ===')
diff = calibrated - prior_matrix
for i in range(5):
    for j in range(5):
        if abs(diff[i][j]) > 0.05:
            direction = '↑' if diff[i][j] > 0 else '↓'
            print(f'  {floor_labels[i]} → {floor_labels[j]}: {direction} {abs(diff[i][j]):.3f}')

## Section 3 — Entry Floor Distribution

Where do visitors start? This determines the initial floor for ABM agents.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ENTRY FLOOR DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════

# First floor mentioned in each review = likely entry point
entry_counts = pd.Series(dtype=int)
for _, row in reviews.iterrows():
    seq = extract_seq(row['text'])
    if seq:
        entry = seq[0]
        entry_counts[entry] = entry_counts.get(entry, 0) + 1

entry_dist = entry_counts / entry_counts.sum()
print('=== Entry Floor Distribution ===')
for f in floor_labels:
    pct = entry_dist.get(f, 0) * 100
    print(f'  {f}: {pct:.1f}%')

# Save
entry_df = pd.DataFrame({'floor': floor_labels, 'entry_prob': [entry_dist.get(f, 0) for f in floor_labels]})
entry_df.to_csv(PROC_DIR / 'entry_distribution.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#00704A', '#1E3932', '#D4E9E2', '#CBA258', '#87CEEB']
ax.bar(floor_labels, [entry_dist.get(f, 0) for f in floor_labels], color=colors, edgecolor='white')
ax.set_title('Entry Floor Distribution (from review sequences)', fontsize=13, fontweight='bold')
ax.set_ylabel('Probability')
for i, f in enumerate(floor_labels):
    ax.text(i, entry_dist.get(f, 0) + 0.01, f'{entry_dist.get(f, 0)*100:.0f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## Section 4 — Save Calibrated Matrix

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════════

calibrated_df.to_csv(PROC_DIR / 'transition_matrix_calibrated.csv')
print(f'Saved: transition_matrix_calibrated.csv')

# Also save the raw counts and alpha for transparency
meta = pd.DataFrame({
    'floor': floor_labels,
    'n_transitions_observed': row_totals.astype(int),
    'alpha_confidence': alpha_per_row.round(4),
    'exit_probability': exit_probs.round(4),
})
meta.to_csv(PROC_DIR / 'calibration_metadata.csv', index=False)
print(f'Saved: calibration_metadata.csv')

print('\n=== Files Created ===')
print('  transition_matrix_calibrated.csv — for ABM agent transitions')
print('  entry_distribution.csv — for ABM agent initial floor')
print('  calibration_metadata.csv — alpha weights and exit probabilities')

print('\n=== Calibrated Matrix (final, for ABM) ===')
display(calibrated_df)

## Transparency Notes

| Component | Source | Confidence |
|---|---|---|
| Physical adjacency prior | Building layout (public) | High |
| Observed transitions | 70 Yelp reviews via NLP | Medium (small sample) |
| Blending weight (alpha) | Sigmoid of observation count, K=10 | Assumption |
| Exit probabilities | 1 - row sum of calibrated matrix | Derived |
| Entry distribution | First floor mentioned in reviews | Medium |

**Key assumption:** K=10 smoothing constant means we need ~10 observations from a floor before we trust the data as much as the prior. This is conservative — with real data, K could be set via cross-validation.